# SpaceX Data Sources and Optional API Collection

## tl;dr
The repeatable analysis uses the included IBM normalized course snapshot. Live API collection is optional and is not claimed as validated by this saved run.

## Context & Methods
IBM course project, reproduced from attributed historical snapshots. See `data/source_manifest.json` for URLs and hashes.

### Key Assumptions
Course success labels include controlled ocean landings; descriptive associations are not causal effects.

## Data
Run from a clean kernel. All ordinary analysis uses included source snapshots.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'falcon9').exists() else Path.cwd().parent
if not (ROOT / 'falcon9').exists():
    raise RuntimeError('Start Jupyter in the repository root or notebooks directory')
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from falcon9.analysis import load_data, group_summary
plt.rcParams.update({'figure.figsize': (9, 4.5), 'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': False})
df = load_data()


### Source provenance

In [2]:
import hashlib, json
manifest = json.loads((ROOT / 'data/source_manifest.json').read_text())
for item in manifest:
    assert hashlib.sha256((ROOT / item['file']).read_bytes()).hexdigest() == item['sha256']
display(pd.DataFrame(manifest)[['file', 'url', 'bytes']])
print('All cached source hashes verified')

All cached source hashes verified


,file,url,bytes
0,data/raw/dataset_part_1.csv,https://cf-courses-data.s3.us.cloud-object-sto...,11634
1,data/raw/dataset_part_2.csv,https://cf-courses-data.s3.us.cloud-object-sto...,12100
2,data/raw/spacex_launch_dash.csv,https://cf-courses-data.s3.us.cloud-object-sto...,2476
3,data/raw/spacex_launch_geo.csv,https://cf-courses-data.s3.us.cloud-object-sto...,7710


## Results
### Inspect the normalized launch snapshot

In [3]:
display(df[['FlightNumber','Date','LaunchSite','Orbit','Outcome']].head())
print(f'{len(df)} unique launches; {df.Date.min().date()} to {df.Date.max().date()}')

90 unique launches; 2010-06-04 to 2020-11-05


,FlightNumber,Date,LaunchSite,Orbit,Outcome
0,1,2010-06-04,CCAFS SLC 40,LEO,None None
1,2,2012-05-22,CCAFS SLC 40,LEO,None None
2,3,2013-03-01,CCAFS SLC 40,ISS,None None
3,4,2013-09-29,VAFB SLC 4E,PO,False Ocean
4,5,2013-12-03,CCAFS SLC 40,GTO,None None


### Optional live API collection
Set `REFRESH_API=True` to request raw launch records. Raw IDs require rocket/payload/core enrichment before comparison with the normalized course snapshot. This notebook does not claim that a fresh raw response recreates the historical dataset.

In [4]:
from falcon9.collection import collect_api
REFRESH_API = False
if REFRESH_API:
    raw_launches = collect_api()
    target = ROOT / 'data/live'
    target.mkdir(parents=True, exist_ok=True)
    raw_launches.to_json(target / 'api_launches.json', orient='records', indent=2)
    display(raw_launches[['id','flight_number','date_utc','rocket']].head())
else:
    print('Live API: NOT RUN. Analysis source: IBM cached normalized CSV, not a fresh API extraction.')

Live API: NOT RUN. Analysis source: IBM cached normalized CSV, not a fresh API extraction.


## Takeaways
Cached source integrity is verified. The live SpaceX service timed out during maintenance; rerun the optional cell in a network-enabled environment to validate live collection. No refreshed API dataset is substituted silently.